In [0]:
# COMMAND ----------
%pip install sarvamai python-dotenv openai sentence-transformers faiss-cpu gradio
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# COMMAND ----------
import os, faiss, json
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

DATABRICKS_TOKEN    = os.environ.get("DATABRICKS_TOKEN")
SARVAM_API_KEY      = os.environ.get("SARVAM_API_KEY")
DATABRICKS_BASE_URL = "https://7474653884759133.ai-gateway.cloud.databricks.com/mlflow/v1"
DATABRICKS_MODEL    = "llama-endpoint"

FAISS_INDEX_PATH  = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/faiss_index.bin"
CHUNK_ID_MAP_PATH = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/chunk_id_map.json"

TABLE_SCHEMES     = "sarkarimitracatalog.sarkaridatabase.schemes_structured"
TABLE_CHUNKS      = "sarkarimitracatalog.sarkaridatabase.scheme_chunks"
TABLE_UNANSWERED  = "sarkarimitracatalog.sarkaridatabase.unanswered_queries"

EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
_embedder   = SentenceTransformer(EMBEDDING_MODEL_NAME)
_faiss_index = faiss.read_index(FAISS_INDEX_PATH)

with open(CHUNK_ID_MAP_PATH, "r") as f:
    _chunk_id_map = json.load(f)

EMPTY_PROFILE = {
    "state": None, "age": None, "gender": None,
    "income_annual": None, "category": None,
    "occupation": None, "has_bpl_card": None,
    "specific_need": None, "clarify_field": None,
}

_faiss_index.ntotal

/local_disk0/.ephemeral_nfs/envs/pythonEnv-dde87ea2-58a4-4312-a3b0-8c8aa02d6070/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


18739

In [0]:
# COMMAND ----------
# ============================================================
# SETUP — Run once at start of each session
# Verifies environment, installs packages, checks tables
# ============================================================

%pip install sarvamai python-dotenv openai sentence-transformers faiss-cpu gradio
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# COMMAND ----------
import os
from dotenv import load_dotenv

load_dotenv()

# ── Token check ────────────────────────────────────────────
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
SARVAM_API_KEY   = os.environ.get("SARVAM_API_KEY")

assert DATABRICKS_TOKEN, "❌ DATABRICKS_TOKEN missing from .env"
assert SARVAM_API_KEY,   "❌ SARVAM_API_KEY missing from .env"

print("✅ Tokens loaded")
print(f"   DATABRICKS_TOKEN : {'set' if DATABRICKS_TOKEN else 'MISSING'}")
print(f"   SARVAM_API_KEY   : {'set' if SARVAM_API_KEY else 'MISSING'}")

✅ Tokens loaded
   DATABRICKS_TOKEN : set
   SARVAM_API_KEY   : set


In [0]:
# COMMAND ----------
# ── Delta table check ──────────────────────────────────────
tables_to_check = [
    "sarkarimitracatalog.sarkaridatabase.schemes_structured",
    "sarkarimitracatalog.sarkaridatabase.scheme_chunks",
]

print("Checking Delta tables...")
all_ok = True
for table in tables_to_check:
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table}").collect()[0]["cnt"]
        print(f"  ✅ {table} — {count:,} rows")
    except Exception as e:
        print(f"  ❌ {table} — MISSING: {e}")
        all_ok = False

if all_ok:
    print("\n✅ All tables present")
else:
    print("\n❌ Fix missing tables before proceeding")

Checking Delta tables...
  ✅ sarkarimitracatalog.sarkaridatabase.schemes_structured — 3,399 rows
  ✅ sarkarimitracatalog.sarkaridatabase.scheme_chunks — 18,739 rows

✅ All tables present


In [0]:
%pip install databricks-vectorsearch


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)
print(vsc.list_endpoints())

{}


In [0]:
# COMMAND ----------
# ── FAISS + Vector Search check ────────────────────────────
import os, faiss, json

FAISS_INDEX_PATH  = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/faiss_index.bin"
CHUNK_ID_MAP_PATH = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/chunk_id_map.json"

print("Checking FAISS files...")
for path in [FAISS_INDEX_PATH, CHUNK_ID_MAP_PATH]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // (1024*1024) if exists else 0
    status = f"✅ {size} MB" if exists else "❌ MISSING"
    print(f"  {status} — {path}")

# Actually load them
_faiss_index = faiss.read_index(FAISS_INDEX_PATH)
with open(CHUNK_ID_MAP_PATH, "r") as f:
    _chunk_id_map = json.load(f)

print(f"\n✅ FAISS index loaded — {_faiss_index.ntotal} vectors")
print(f"✅ Chunk map loaded — {len(_chunk_id_map)} entries")

Checking FAISS files...
  ✅ 27 MB — /Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/faiss_index.bin
  ✅ 2 MB — /Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/chunk_id_map.json

✅ FAISS index loaded — 18739 vectors
✅ Chunk map loaded — 2 entries


In [0]:
# COMMAND ----------
# ── Load embedding model ───────────────────────────────────
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
_embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Embedding model loaded — {EMBEDDING_MODEL_NAME}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-dde87ea2-58a4-4312-a3b0-8c8aa02d6070/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded — paraphrase-multilingual-MiniLM-L12-v2


In [0]:
# COMMAND ----------
import os, faiss, json

FAISS_INDEX_PATH  = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/faiss_index.bin"
CHUNK_ID_MAP_PATH = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/chunk_id_map.json"

_faiss_index = faiss.read_index(FAISS_INDEX_PATH)
with open(CHUNK_ID_MAP_PATH, "r") as f:
    _chunk_id_map = json.load(f)

_faiss_index.ntotal

18739

In [0]:
# COMMAND ----------
import os

faiss_path     = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/faiss_index.bin"
chunk_map_path = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/chunk_id_map.json"

results = []
for path in [faiss_path, chunk_map_path]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // (1024*1024) if exists else 0
    results.append({"path": path, "exists": exists, "size_mb": size})

display(spark.createDataFrame(results))

exists,path,size_mb
true,/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/faiss_index.bin,27
true,/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/chunk_id_map.json,2


In [0]:
# COMMAND ----------
# ── RAM check ──────────────────────────────────────────────
import psutil

mem = psutil.virtual_memory()
available_gb = mem.available / 1e9
total_gb     = mem.total / 1e9
used_pct     = mem.percent

print(f"Memory status:")
print(f"  Total     : {total_gb:.1f} GB")
print(f"  Available : {available_gb:.1f} GB")
print(f"  Used      : {used_pct:.0f}%")

if available_gb < 4:
    print("⚠️  WARNING: Less than 4GB free — may OOM during model load")
elif available_gb < 6:
    print("⚠️  Tight on memory — close other notebooks before running app")
else:
    print("✅ Memory looks fine")

print("\n✅ Setup check complete — ready to run intelligence notebooks")

Memory status:
  Total     : 16.4 GB
  Available : 6.0 GB
  Used      : 64%
⚠️  Tight on memory — close other notebooks before running app

✅ Setup check complete — ready to run intelligence notebooks
